In [ ]:
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By

In [ ]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

In [ ]:
s = Service("chromedriver.exe")
driver = webdriver.Chrome(service=s)
driver.get("https://www.smartprix.com/mobile_headphones")
time.sleep(2)

driver.find_element(by=By.XPATH, value='//*[@id="app"]/main/aside/div/div[6]/div[2]/label[2]/input').click()
time.sleep(2)

old_height = driver.execute_script("return document.body.scrollHeight")
same_count = 0

while True:

    driver.find_element(by=By.XPATH, value='//*[@id="app"]/main/div[1]/div[2]/div/div[3]').click()
    time.sleep(5)
    new_height = driver.execute_script("return document.body.scrollHeight")

    print("old_height:", old_height)
    print("new_height:", new_height)

    if new_height == old_height:
        same_count += 1
        if same_count >= 3:  # stop only after 3 unchanged checks in a row
            break
    else:
        same_count = 0

    old_height = new_height

html = driver.page_source

with open("earbuds.html", "w", encoding="utf-8") as file:
    file.write(html)

input("Press Enter in the terminal to close the browser...")
driver.quit()

In [ ]:
with open("earbuds.html", "r", encoding="utf-8") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")
soup.prettify()
container = soup.find_all('div', {'class': 'sm-product has-tag has-features has-actions'})
test = container[0]
product_name = []
price = []
rating = []
design_type = []
fit_style = []
connectivity = []
bluetooth_version = []
has_anc = []
has_microphone = []
wireless_range = []
battery_life = []

for item in container:
    try:
        product_name.append(
            item.find('a', {'class': 'name clamp-2'}).find('h2').text
        )
    except:
        product_name.append('Not mentioned')

    try:
        price.append(item.find('span', {'class': 'price'}).text)
    except:
        price.append(np.nan)

    try:
        rating.append(item.find('div', {'class': 'rating'}).find('span')['style'])
    except:
        rating.append('Not mentioned')

    specs = item.find('ul', {'class': 'sm-feat specs'}).find_all('li')
    try:
        design_type.append(specs[0].text)
    except:
        design_type.append('Not mentioned')

    try:
        fit_style.append(specs[1].text)
    except:
        fit_style.append('Not mentioned')

    try:
        connectivity.append(specs[2].text)
    except:
        connectivity.append('Not mentioned')

    try:
        bluetooth_version.append(specs[3].text)
    except:
        bluetooth_version.append('Not mentioned')

    try:
        has_anc.append(specs[4].text)
    except:
        has_anc.append('Not mentioned')

    try:
        has_microphone.append(specs[5].text)
    except:
        has_microphone.append('Not mentioned')

    try:
        wireless_range.append(specs[6].text)
    except:
        wireless_range.append('Not mentioned')

    try:
        battery_life.append(specs[7].text)
    except:
        battery_life.append('Not mentioned')

In [ ]:
df = pd.DataFrame({
    'product_name': product_name,
    'price': price,
    'rating': rating,
    'design_type': design_type,
    'fit_style': fit_style,
    'connectivity': connectivity,
    'bluetooth_version': bluetooth_version,
    'anc': has_anc,
    'microphone': has_microphone,
    'wireless_range': wireless_range,
    'battery_life': battery_life
})

In [ ]:
df.to_csv("earbuds.csv", index=False)